# Week 63, Day 4: Metaclasses

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Dog:
    sound = "woof"
    def speak(self):
        return self.sound

print(type(Dog()), type(Dog), type(type))

Cat = type("Cat", (), {"sound": "miaow", "speak": lambda self: self.sound})
print(Cat().speak(), Cat.__name__, isinstance(Cat, type))

In [ ]:
def make_point_class(*fields):
    def __init__(self, *values):
        for f, v in zip(fields, values):
            setattr(self, f, v)
    def __repr__(self):
        inner = ", ".join(f"{f}={getattr(self, f)!r}" for f in fields)
        return f"{type(self).__name__}({inner})"
    return type("Point", (), {"__init__": __init__, "__repr__": __repr__, "fields": fields})

Point3 = make_point_class("x", "y", "z")
p = Point3(1, 2, 3)
print(p, Point3.fields, type(Point3).__name__)
print(issubclass(type, object), isinstance(object, type), type(object))

In [ ]:
class Meta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        print("1. prepare", name, kwargs)
        return {}

    def __new__(mcls, name, bases, ns, **kwargs):
        print("3. new", name, sorted(k for k in ns if not k.startswith("__")))
        return super().__new__(mcls, name, bases, ns)

    def __init__(cls, name, bases, ns, **kwargs):
        print("5. init", name)
        super().__init__(name, bases, ns)

class Base(metaclass=Meta):
    def __init_subclass__(cls, **kwargs):
        print("4. init_subclass", cls.__name__, kwargs)

class Named:
    def __set_name__(self, owner, name):
        print("4. set_name", name)

class Item(Base, flag=True):
    print("2. body runs")
    price = Named()

In [ ]:
class Validated(type):
    def __new__(mcls, name, bases, ns):
        for key, value in ns.items():
            if callable(value) and not key.startswith("_") and not value.__doc__:
                raise TypeError(f"{name}.{key} needs a docstring")
        ns["created_by"] = mcls.__name__          # add an attribute to every class
        return super().__new__(mcls, name, bases, ns)

class Service(metaclass=Validated):
    def start(self):
        "Start the service."

print(Service.created_by)
try:
    class Broken(Service):                          # the metaclass is inherited
        def stop(self):
            pass
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Counted(type):
    def __init__(cls, name, bases, ns):
        super().__init__(name, bases, ns)
        cls._instances = []

    def __call__(cls, *args, **kwargs):
        obj = super().__call__(*args, **kwargs)     # the normal __new__ + __init__
        cls._instances.append(obj)
        return obj

    def __len__(cls):
        return len(cls._instances)

    def __iter__(cls):
        return iter(cls._instances)

    @property
    def newest(cls):
        return cls._instances[-1] if cls._instances else None

class Order(metaclass=Counted):
    def __init__(self, number):
        self.number = number

Order(1), Order(2), Order(3)
print(len(Order), [o.number for o in Order], Order.newest.number)
print(hasattr(Order(4), "newest"), len(Order))

In [ ]:
class Plugin:
    registry = {}
    def __init_subclass__(cls, key=None, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[key or cls.__name__.lower()] = cls

class Csv(Plugin, key="csv"):
    pass
class Json(Plugin):
    pass
print(sorted(Plugin.registry))

class MetaA(type):
    pass
class MetaB(type):
    pass
class A(metaclass=MetaA):
    pass
class B(metaclass=MetaB):
    pass
try:
    class C(A, B):
        pass
except TypeError as e:
    print("TypeError:", str(e)[:18])

> **Key idea.** Classes are objects whose type is a metaclass, normally type; type(name, bases, ns) makes a class. The class statement picks the metaclass, calls __prepare__ for the namespace, runs the body in it, then calls the metaclass, whose type.__new__ calls __set_name__ on descriptors and __init_subclass__ on the base; decorators run last. A metaclass subclasses type: __new__ can change the namespace before the class exists, __init__ acts on the finished class, and both apply to every subclass. Methods on the metaclass belong to classes: __call__ controls instance creation, and __len__, __iter__ and properties give classes behaviour. Prefer __init_subclass__, class decorators and descriptors when they suffice; unrelated metaclasses conflict.

### Exercise: Classes from type()

Write `make_record(name, fields)` that uses `type(name, bases, ns)` (no `class` statement) to build a class whose `__init__` takes one argument per field, in order, as positional or keyword arguments; whose `__repr__` is like `Book(title='Aké', year=1981)`; whose `__eq__` compares class and field values; and which has a class attribute `fields` (a tuple). Then `subclass(base, name, **attrs)`: a new subclass of `base` with the given class attributes, again with `type`.

In [ ]:
def make_record(name, fields):
    ___

def subclass(base, name, **attrs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_maketype():\n    \"type() makes classes\"\n    Book = make_record(\"Book\", [\"title\", \"year\"])\n    b = Book(\"Ak\u00e9\", year=1981)\n    assert b.title == \"Ak\u00e9\" and b.year == 1981 and repr(b) == \"Book(title='Ak\u00e9', year=1981)\"\n    assert Book.__name__ == \"Book\" and Book.fields == (\"title\", \"year\") and type(Book) is type\n    assert b == Book(\"Ak\u00e9\", 1981) and b != Book(\"Ak\u00e9\", 1982)\n    Novel = subclass(Book, \"Novel\", genre=\"fiction\")\n    n = Novel(\"Ake\", 1981)\n    assert isinstance(n, Book) and Novel.genre == \"fiction\" and repr(n) == \"Novel(title='Ake', year=1981)\"\n    assert not uses(source, \"class\")")

### Exercise: A registering metaclass

Write a metaclass `Registered` that records every class it creates in the class-level dict `Registered.classes`, keyed by the lower-cased class name, **except** classes that set `abstract = True` in their own body. A duplicate name raises `TypeError` at class creation. Then define a base class `Handler` with `Registered` as its metaclass and `abstract = True`.

In [ ]:
class Registered(type):
    classes = {}
    ___

class Handler(metaclass=Registered):
    abstract = True

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_registry():\n    \"metaclass registration\"\n    class Email(Handler):\n        pass\n    class Sms(Handler):\n        pass\n    class Special(Handler):\n        abstract = True\n    class Push(Special):\n        pass\n    assert {\"email\", \"sms\", \"push\"} <= set(Registered.classes) and \"special\" not in Registered.classes\n    assert \"handler\" not in Registered.classes and Registered.classes[\"email\"] is Email\n    assert type(Push) is Registered\n    try:\n        class EMAIL(Handler):\n            pass\n        assert False\n    except TypeError:\n        pass")

### Exercise: One instance only

Write a metaclass `Single` whose classes have at most one instance: the first call creates it normally (through `super().__call__`), later calls return the same object without running `__init__` again. Each class (including subclasses) gets its own instance. Add a metaclass method `reset(cls)` that forgets the class's instance.

In [ ]:
class Single(type):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_singleton():\n    \"controlling __call__\"\n    inits = []\n    class Config(metaclass=Single):\n        def __init__(self, path=\"app.cfg\"):\n            inits.append(path)\n            self.path = path\n    a = Config()\n    b = Config(\"other.cfg\")\n    assert a is b and a.path == \"app.cfg\" and inits == [\"app.cfg\"]\n    class Child(Config):\n        pass\n    c = Child(\"child.cfg\")\n    assert c is not a and Child() is c and isinstance(c, Config)\n    Config.reset()\n    d = Config(\"new.cfg\")\n    assert d is not a and d.path == \"new.cfg\" and Child() is c\n    assert not hasattr(d, \"reset\")")

### Exercise: Checking subclasses as they're made

Write a metaclass `Contract`: a class may list method names in `requires = (...)`. Every class made by `Contract` that **isn't** marked `partial = True` in its own body must have a callable attribute (its own or inherited) for every name required by **any** class in its MRO; otherwise class creation raises `TypeError("Name is missing: a, b")` with the missing names sorted. Also give every class a sorted tuple `all_requirements`.

In [ ]:
class Contract(type):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_interface():\n    \"checks at class creation\"\n    class Storage(metaclass=Contract):\n        requires = (\"read\", \"write\")\n        partial = True\n    class Cached(Storage):\n        requires = (\"clear\",)\n        partial = True\n    class Disk(Cached):\n        def read(self): pass\n        def write(self): pass\n        def clear(self): pass\n    assert Disk.all_requirements == (\"clear\", \"read\", \"write\")\n    try:\n        class Half(Cached):\n            def read(self): pass\n        assert False\n    except TypeError as e:\n        assert str(e) == \"Half is missing: clear, write\"\n    class Better(Disk):\n        pass\n    assert Better.all_requirements == (\"clear\", \"read\", \"write\")\n    try:\n        class Odd(Storage):\n            read = write = 5\n        assert False\n    except TypeError as e:\n        assert str(e) == \"Odd is missing: read, write\"")

### Exercise: No accidental redefinitions

Defining two methods with the same name in one class body is a common mistake: the second silently replaces the first. Write a metaclass `NoRepeats` whose `__prepare__` returns a dict subclass that raises `TypeError("name defined twice: X")` when a name is assigned a **second** time in the class body. Dunder names (starting and ending with `__`) may be reassigned. Properties built with `@x.setter` reuse the name on purpose, so allow a reassignment when **both** the old and the new value are `property` objects.

In [ ]:
class NoRepeats(type):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_prepare():\n    \"__prepare__\"\n    class Fine(metaclass=NoRepeats):\n        x = 1\n        def a(self):\n            return \"a\"\n        @property\n        def p(self):\n            return self._p\n        @p.setter\n        def p(self, v):\n            self._p = v\n    f = Fine()\n    f.p = 3\n    assert f.p == 3 and Fine().a() == \"a\" and type(Fine) is NoRepeats\n    try:\n        class Broken(metaclass=NoRepeats):\n            def save(self):\n                return 1\n            def save(self):\n                return 2\n        assert False\n    except TypeError as e:\n        assert str(e) == \"name defined twice: save\"\n    try:\n        class AlsoBroken(metaclass=NoRepeats):\n            limit = 1\n            limit = 2\n        assert False\n    except TypeError:\n        pass\n    class Inherits(Fine):\n        def a(self):\n            return \"overridden\"\n    assert Inherits().a() == \"overridden\"")

### Bug hunt: A metaclass that forgot to return

The metaclass's `__new__` creates the class but doesn't return it, so the class statement binds `Thing` to `None` and `Thing.__name__` fails with `AttributeError`. Return the class so this prints `Thing`.

In [ ]:
class Meta(type):
    def __new__(mcls, name, bases, ns):
        cls = super().__new__(mcls, name, bases, ns)
        cls.tag = name.lower()

class Thing(metaclass=Meta):
    pass

print(Thing.__name__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"return the class\"\n    expect_output(\"Thing\")")

### Bug hunt: Two metaclasses that don't agree

`Mixed` inherits from classes with unrelated metaclasses, so Python can't choose one and raises `TypeError` ("metaclass conflict"). Make `LoggedMeta` a subclass of `TrackedMeta`, so it's the most derived one, and this prints `ok`.

In [ ]:
class TrackedMeta(type):
    pass

class LoggedMeta(type):
    pass

class Tracked(metaclass=TrackedMeta):
    pass

class Logged(metaclass=LoggedMeta):
    pass

class Mixed(Tracked, Logged):
    pass

print("ok")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"compatible metaclasses\"\n    expect_output(\"ok\")")

### Bug hunt: A singleton that calls itself

`__call__` creates the instance with `cls()`, which is itself a call on the class, which runs `__call__` again, until `RecursionError`. Create it with `super().__call__(...)` so this prints `True`.

In [ ]:
class OnlyOne(type):
    def __call__(cls, *args, **kwargs):
        if not hasattr(cls, "_one"):
            cls._one = cls(*args, **kwargs)
        return cls._one

class Logger(metaclass=OnlyOne):
    pass

print(Logger() is Logger())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"super().__call__\"\n    expect_output(\"True\")")

### Puzzle: A metaclass that adds an attribute

Arrange the lines of a metaclass that stamps every class with its own name in lower case. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # type(type) is type: True or False
q = ___   # the metaclass of an ordinary class, as a name: "type" or "object"
r = ___   # True or False: a metaclass is inherited by subclasses
s = ___   # which runs first: "__prepare__" or "the class body"
t = ___   # True or False: methods defined on a metaclass can be called on instances of its classes

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q == \"type\" and r is True, \"p, q and r\"\n    assert s == \"__prepare__\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a choices metaclass

Write a metaclass `ChoiceMeta` and a base class `Choice` so that

```
class Colour(Choice):
    RED = "r"
    GREEN = "g"
```

makes each upper-case class attribute (in the body, not starting with `_`) a **member**: an instance of `Colour` with `name` and `value`. Then:

- `Colour.RED` is the member; `repr` is `<Colour.RED: 'r'>`, `str` is `Colour.RED`;
- `Colour("g")` looks a member up **by value** (a `ValueError` for unknown values), and returns the existing member: no new instances can be made;
- `list(Colour)` gives members in definition order, `len(Colour)` their number, and `"r" in Colour` / `Colour.RED in Colour` test values and members;
- duplicate values raise `TypeError` at class creation;
- `Colour.RED = "x"` and `del Colour.RED` raise `AttributeError` (define `__setattr__` and `__delattr__` on the metaclass);
- members compare by identity and are hashable; other attributes (methods, lower-case names) stay ordinary, and members can use methods defined in the class body;
- subclassing a choice that has members raises `TypeError`, but `Choice` itself (with no members) can be subclassed.

In [ ]:
class ChoiceMeta(type):
    ___

class Choice(metaclass=ChoiceMeta):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_members():\n    \"members and lookup\"\n    class Colour(Choice):\n        RED = \"r\"\n        GREEN = \"g\"\n        BLUE = \"b\"\n        def describe(self):\n            return f\"{self.name.lower()} ({self.value})\"\n    assert isinstance(Colour.RED, Colour) and Colour.RED.name == \"RED\" and Colour.RED.value == \"r\"\n    assert repr(Colour.GREEN) == \"<Colour.GREEN: 'g'>\" and str(Colour.BLUE) == \"Colour.BLUE\"\n    assert Colour(\"g\") is Colour.GREEN and Colour.BLUE.describe() == \"blue (b)\"\n    assert list(Colour) == [Colour.RED, Colour.GREEN, Colour.BLUE] and len(Colour) == 3\n    assert \"r\" in Colour and Colour.RED in Colour and \"x\" not in Colour\n    assert {Colour.RED: 1}[Colour(\"r\")] == 1\n    try:\n        Colour(\"x\")\n        assert False\n    except ValueError:\n        pass\n\ndef test_rules():\n    \"the rules\"\n    class Size(Choice):\n        SMALL = 1\n        LARGE = 3\n        default_note = \"sizes\"\n    assert Size.default_note == \"sizes\" and len(Size) == 2\n    for action in (lambda: setattr(Size, \"SMALL\", 2), lambda: delattr(Size, \"SMALL\")):\n        try:\n            action()\n            assert False\n        except AttributeError:\n            pass\n    try:\n        class Bad(Choice):\n            A = 1\n            B = 1\n        assert False\n    except TypeError:\n        pass\n    try:\n        class More(Size):\n            HUGE = 9\n        assert False\n    except TypeError:\n        pass\n    assert Size.SMALL is Size(1) and Size.SMALL != Size.LARGE")

## Warm-up

A class is an instance of `type`; `type` is an instance of itself. And calling `type` with three arguments makes a class, just as a `class` statement does.

## Classes are objects

Everything in Python is an object, including classes: you can pass them to functions, store them in lists and dicts, add attributes to them, and create them at run time. Every object has a type, and the type of an ordinary class is **`type`**. Since `type` makes classes, it's called a **metaclass**: a class whose instances are classes.

`type` does two jobs depending on how it's called:

- `type(obj)` returns the type of `obj`;
- `type(name, bases, namespace)` **creates a class**: `name` is a string, `bases` a tuple of base classes, and `namespace` a dict of the class's attributes and methods.

So the relationships are: an instance's type is its class (`type(dog) is Dog`), a class's type is its metaclass (`type(Dog) is type`), and every class, metaclasses included, inherits from `object` eventually.

## What the class statement does

When Python runs

```
class Item(Base, metaclass=Meta, flag=True):
    price = Field()
    def show(self): ...
```

it does this:

1. **Pick the metaclass**: the explicit `metaclass=` if given, otherwise the metaclass of the bases (normally `type`). If bases have different metaclasses, it uses the most derived one, or raises `TypeError` ("metaclass conflict") if none of them is a subclass of all the others.
2. **Prepare the namespace**: `ns = Meta.__prepare__("Item", bases, flag=True)`, normally an empty dict.
3. **Run the class body** with `ns` as its local namespace: every assignment and `def` in the body becomes an entry in `ns` (plus `__module__` and `__qualname__`).
4. **Create the class**: `Item = Meta("Item", bases, ns, flag=True)`. That call runs `Meta.__new__` (which, through `type.__new__`, makes the class object) and then `Meta.__init__`.
5. Inside `type.__new__`, Python calls **`__set_name__`** on every descriptor in the namespace (Monday) and then **`__init_subclass__`** on the nearest base class (Week 35), passing the extra keywords (`flag=True`).
6. Finally, any **class decorators** run (tomorrow), and the result is bound to the name `Item`.

## Writing a metaclass

A metaclass is a subclass of `type`. Its methods receive the **class** where ordinary methods receive an instance, so by convention the first parameter is called `mcls` (in `__new__`, the metaclass) or `cls` (in other methods, the class being made).

- Override **`__new__(mcls, name, bases, ns, **kwargs)`** to inspect or **change** the namespace before the class exists (add methods, rename attributes, reject bad definitions), and return `super().__new__(mcls, name, bases, ns)`.
- Override **`__init__(cls, name, bases, ns, **kwargs)`** to act on the finished class (register it, check it).
- A metaclass is **inherited**: every subclass of a class with `metaclass=Meta` is also made by `Meta`, so the checks apply to the whole hierarchy. That's the main thing a metaclass gives that a decorator doesn't.

## Metaclass methods: behaviour for classes

Methods defined on a metaclass are methods **of the class** (they aren't visible on instances). The same lookup rules as yesterday apply one level up: `Cls.attr` looks in the metaclass for data descriptors, then in the class and its bases, then in the metaclass again. So a metaclass can give classes:

- **special methods**: `__iter__`, `__len__`, `__getitem__`, `__contains__` and `__repr__` on the metaclass make `for x in Cls`, `len(Cls)` and `Cls[key]` work (this is how `enum.Enum` classes can be iterated);
- **properties of the class**, defined as properties on the metaclass;
- **`__call__`**: `Cls(...)` is a call on the class object, so it runs `type(Cls).__call__`, which normally calls `__new__` and `__init__`. Overriding it controls instance creation: singletons, caching instances, counting them, or returning existing objects.

## When not to use one

Metaclasses are powerful and rarely necessary. Before writing one, check the simpler tools, in this order:

- **`__init_subclass__`** (Week 35): runs for each new subclass, can take keyword arguments, and can register, check or modify the subclass. It covers most "do something whenever a subclass is defined" needs.
- **A class decorator** (tomorrow): runs once on one class and can change it or return a replacement. Explicit, and doesn't affect subclasses unless they're decorated too.
- **Descriptors with `__set_name__`** (Monday): per-attribute behaviour.

What only a metaclass can do: change the namespace **before** the class is created (`__prepare__`, `__new__`), control what calling the class does for a whole hierarchy (`__call__`), give the class itself special methods (`len(Cls)`), and apply all of that **automatically** to every subclass.

The costs: metaclasses are surprising to readers, and two libraries with different metaclasses can't easily be combined, because a class can't have two unrelated metaclasses (the conflict in step 1). Python's own uses (`abc.ABCMeta`, `enum.EnumMeta`) are good models of when the cost is worth it.

## Practice

## Quiz

## Daily challenge

## Before you go

Look at `type(enum.Enum)` and `type(abc.ABC)` in a Python shell: both are metaclasses from the standard library. Tomorrow: class decorators, the simpler tool for changing one class at a time.